# Uncertainty and selective prediction

Question: uncertainty and selective prediction for explicitly expressed activities.

All examples are fictional. Run cells in order after installing the project.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
if not (root / 'src').exists():
    raise RuntimeError('Run from repository root or notebooks directory')
sys.path.insert(0, str(root / 'src'))

## Experiment
Inspect the intermediate outputs and check the assumptions before interpreting them.

In [ ]:
from socialhar.synthetic import generate
from socialhar.split import grouped_split
from socialhar.models import baseline, predict_selective
from socialhar.evaluation import cluster_bootstrap_f1, reliability
train, test = grouped_split(generate())
model = baseline().fit(train.text, train.label)
print(cluster_bootstrap_f1(test, model.predict(test.text), repeats=100))
print(reliability(test.label, model.predict_proba(test.text), model.classes_))

In [ ]:
import pandas as pd
rows = []
for threshold in [0, .3, .5, .7, .9, 1]:
    results = predict_selective(model, test.text, threshold)
    kept = [i for i, result in enumerate(results) if result['label'] != 'abstain']
    accuracy = sum(results[i]['label'] == test.iloc[i].label for i in kept) / len(kept) if kept else None
    rows.append({'threshold': threshold, 'coverage': len(kept) / len(test), 'selective_accuracy': accuracy})
pd.DataFrame(rows)

## Interpretation and limitations
Choose an abstention threshold using a validation partition, never the test set. Softmax or logistic confidence is not automatically calibrated. A synthetic bootstrap interval only captures variability in this toy generator.

## Research extension
Repeat this analysis on an independently annotated, eligible dataset. Record the sampling frame, annotation disagreements, seed, model version and failure cases.